In [1]:
import numpy as np
import yancc
from yancc.field import Field
from yancc.velocity_grids import MaxwellSpeedGrid, UniformPitchAngleGrid
from yancc.species import LocalMaxwellian, GlobalMaxwellian
from yancc.solve import solve_dke
from yancc.misc import normalize_fluxes_sfincs
from yancc import yancctools


In [3]:
from yancc.yancctools.yancctools_test import _temperature_profile, _density_profile

nx = 5 # resolution in x/ speed coordinate
na = 65 # resolution in  pitch angle coordinate
nt = 17 # resolution in theta / poloidal angle
nz = 33 # resolution in zeta / toroidal angle

speedgrid = MaxwellSpeedGrid(nx)
pitchgrid = UniformPitchAngleGrid(na)

import desc
eq = desc.examples.get("W7-X")
rho = 0.5 # surface label
field = Field.from_desc(eq, rho, nt, nz)

# Define global species (profiles vs rho)
global_species = [
    GlobalMaxwellian(
        yancc.species.Hydrogen,
        temperature=_temperature_profile,
        density=_density_profile,
    ),
    GlobalMaxwellian(
        yancc.species.Electron,
        temperature=_temperature_profile,
        density=_density_profile,
    )
]

# Localize for a single surface check
species = [s.localize(field.rho) for s in global_species]

### Single Surface Ambipolar Search

In [ ]:
## 1. Run the physics calculations
erho_grid, flux_diffs, roots = yancctools.find_ambipolar_roots(
    field, pitchgrid, speedgrid, species, erho_min=-10000, erho_max=10000, n_points=11
)

# 2. Visualize the results
yancctools.plot_ambipolar_scan(erho_grid, flux_diffs, roots)

### Radial Ambipolar Profile Scan

In [ ]:
# 1. Define radial grid
rho_grid = np.linspace(0.2, 0.8, 5)

# 2. Run radial scan in parallel
results = yancctools.scan_ambipolar_profile(
    rho_grid,
    eq_type="desc",
    eq_data=eq,
    nt=nt, 
    nz=nz,
    pitchgrid=pitchgrid,
    speedgrid=speedgrid,
    global_species=global_species,
    num_processors=4,
    rtol=1e-4
)

# 3. Plot the profile
yancctools.plot_ambipolar_profile(results)

Starting radial scan over 5 surfaces with 4 processors...
